# TF-IDF Feature Engineering and Classical ML Baselines

This notebook builds the TF-IDF feature representation and trains
four classical machine learning models on the WELFake dataset.

The pipeline follows a two-stage approach:

Stage 1 -- TF-IDF parameter selection:
Three controlled experiments identify the optimal vocabulary size,
n-gram range, and document frequency thresholds. Each experiment
uses Logistic Regression with 3-fold cross validation on the
training set only.

Stage 2 -- Classifier hyperparameter tuning:
GridSearchCV with 5-fold stratified cross validation finds the
optimal regularization parameter C for Logistic Regression and
LinearSVC. Random Forest and XGBoost use standard defaults as
established in Alghamdi et al. (2022).

The test set is held out completely and used only for final
evaluation in Section 6.

## Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
import os
import time
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('..')

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix

from src.features import (
    build_tfidf,
    experiment_max_features,
    experiment_ngram_range,
    experiment_df_params,
    run_gridsearch,
    save_model,
    load_model
)
from src.evaluate import (
    compute_metrics,
    build_results_table,
    save_results
)
from src.utils import set_seed, print_section, ensure_dirs

set_seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

PROCESSED_DIR = '../data/processed/'
MODELS_DIR    = '../models/'
FIGURES_DIR   = '../outputs/figures/features/'
RESULTS_DIR   = '../outputs/results/'

ensure_dirs([MODELS_DIR, FIGURES_DIR, RESULTS_DIR])

## Section 1 -- Load Processed Data

The three splits saved during preprocessing are loaded here.
Only the content and label columns are needed for this notebook.
Shape and class balance are printed to confirm the data loaded
correctly before any processing begins.

In [ ]:
print_section('Loading Processed Data')

train_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'train_clean.csv'))
val_df   = pd.read_csv(os.path.join(PROCESSED_DIR, 'val_clean.csv'))
test_df  = pd.read_csv(os.path.join(PROCESSED_DIR, 'test_clean.csv'))

X_train_text = train_df['content']
X_val_text   = val_df['content']
X_test_text  = test_df['content']

y_train = train_df['label']
y_val   = val_df['label']
y_test  = test_df['label']

print(f'Train : {len(train_df):,} articles')
print(f'Val   : {len(val_df):,} articles')
print(f'Test  : {len(test_df):,} articles')
print()
print('Class balance -- Train:')
print(y_train.value_counts(normalize=True).round(4))

## Section 2 -- Stage 1: TF-IDF Parameter Experiments

### Experiment A -- Optimal Vocabulary Size (max_features)

The vocabulary size controls how many terms are retained from
the full corpus vocabulary. Too few terms lose discriminative
signal. Too many terms add noise and increase memory usage
without meaningful accuracy gains.

A Logistic Regression model is trained at four vocabulary sizes
using 3-fold cross validation. The F1 macro score at each size
identifies the point where adding more features stops helping.
This elbow point becomes the optimal max_features setting.

In [ ]:
print_section('Experiment A: max_features Selection')
print('Training LR at 4 vocabulary sizes using 3-fold CV.')
print('This may take 10-15 minutes on HPCC.')
print()

results_features = experiment_max_features(
    X_train_text, y_train,
    feature_counts=[10000, 20000, 30000, 50000]
)

print('\nExperiment A Results:')
print(results_features.to_string(index=False))

### Experiment A -- Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    results_features['max_features'],
    results_features['mean_f1'],
    marker='o', color='#2A9D8F',
    linewidth=2, markersize=8,
    label='Mean F1 macro'
)

ax.fill_between(
    results_features['max_features'],
    results_features['mean_f1'] - results_features['std_f1'],
    results_features['mean_f1'] + results_features['std_f1'],
    alpha=0.2, color='#2A9D8F',
    label='Std deviation'
)

best_idx = results_features['mean_f1'].idxmax()
best_row = results_features.loc[best_idx]
ax.axvline(
    x=best_row['max_features'],
    color='#E76F51', linestyle='--', alpha=0.8,
    label=f"Optimal: {int(best_row['max_features']):,} features"
)

ax.set_xlabel('Number of TF-IDF Features', fontsize=12)
ax.set_ylabel('F1 Macro (3-fold CV)', fontsize=12)
ax.set_title('F1 Score vs Vocabulary Size', fontsize=14)
ax.xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, p: f'{int(x):,}'))
ax.legend()
plt.tight_layout()
plt.savefig(
    os.path.join(FIGURES_DIR, '10_feature_count_experiment.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()

OPTIMAL_MAX_FEATURES = int(best_row['max_features'])
print(f'Optimal max_features: {OPTIMAL_MAX_FEATURES:,}')

### Experiment B -- N-gram Range Selection

Unigrams capture individual word signals. Adding bigrams
captures two-word phrase patterns such as "white house"
or "fake news" which carry meaning beyond their individual
words. This experiment tests whether bigrams improve
classification over unigrams alone.

In [ ]:
print_section('Experiment B: N-gram Range Selection')

results_ngram = experiment_ngram_range(
    X_train_text, y_train,
    ngram_options=[(1, 1), (1, 2)]
)

print('\nExperiment B Results:')
print(results_ngram.to_string(index=False))

best_ngram_idx = results_ngram['mean_f1'].idxmax()
OPTIMAL_NGRAM  = results_ngram.loc[best_ngram_idx, 'ngram_range']
if isinstance(OPTIMAL_NGRAM, str):
    OPTIMAL_NGRAM = eval(OPTIMAL_NGRAM)
OPTIMAL_NGRAM = tuple(OPTIMAL_NGRAM)
print(f'\nOptimal ngram_range: {OPTIMAL_NGRAM}')

### Experiment C -- Document Frequency Thresholds

min_df removes terms appearing in fewer than min_df documents,
which eliminates rare noise terms like typos. max_df removes
terms appearing in more than max_df proportion of documents,
which eliminates uninformative high-frequency terms that
appear across nearly all articles regardless of class.

In [ ]:
print_section('Experiment C: min_df and max_df Selection')

results_df_params = experiment_df_params(X_train_text, y_train)

print('\nExperiment C Results:')
print(results_df_params.to_string(index=False))

best_df_idx    = results_df_params['mean_f1'].idxmax()
OPTIMAL_MIN_DF = int(results_df_params.loc[best_df_idx, 'min_df'])
OPTIMAL_MAX_DF = float(results_df_params.loc[best_df_idx, 'max_df'])
print(f'\nOptimal min_df: {OPTIMAL_MIN_DF}')
print(f'Optimal max_df: {OPTIMAL_MAX_DF}')

## Section 3 -- Build Final TF-IDF Matrix

Using the optimal parameters identified across all three
experiments, the final TF-IDF vectorizer is fit on the
training set and applied to all three splits.

The fitted vectorizer is saved to models/ for two reasons:
first, to enable consistent transformation in later notebooks
that build the hybrid model; second, to transform new articles
in the Streamlit demonstration application.

In [ ]:
print_section('Building Final TF-IDF Matrix')

print('Optimal TF-IDF configuration:')
print(f'  max_features : {OPTIMAL_MAX_FEATURES:,}')
print(f'  ngram_range  : {OPTIMAL_NGRAM}')
print(f'  min_df       : {OPTIMAL_MIN_DF}')
print(f'  max_df       : {OPTIMAL_MAX_DF}')
print(f'  sublinear_tf : True')
print()

vectorizer, X_train, X_val, X_test = build_tfidf(
    X_train_text, X_val_text, X_test_text,
    max_features=OPTIMAL_MAX_FEATURES,
    ngram_range=OPTIMAL_NGRAM,
    min_df=OPTIMAL_MIN_DF,
    max_df=OPTIMAL_MAX_DF,
    sublinear_tf=True
)

save_model(vectorizer,
    os.path.join(MODELS_DIR, 'tfidf_vectorizer.joblib'))

## Section 4 -- Stage 2: Classifier Hyperparameter Tuning

GridSearchCV with 5-fold stratified cross validation finds
the optimal regularization strength C for Logistic Regression
and LinearSVC. All tuning uses only the training set.

The regularization parameter C controls the trade-off between
fitting the training data and keeping model weights small.
Low C applies strong regularization producing a simpler model.
High C allows the model to fit training data more closely.

In [ ]:
print_section('GridSearchCV - Logistic Regression')

lr_base = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    solver='saga',
    random_state=42
)

lr_best, lr_best_params, lr_best_score, lr_cv_results = run_gridsearch(
    X_train, y_train,
    model=lr_base,
    param_grid={'C': [0.01, 0.1, 1.0, 10.0]},
    model_name='Logistic Regression',
    cv=5
)

print(f'\nBest C     : {lr_best_params["C"]}')
print(f'Best CV F1 : {lr_best_score:.4f}')

### LinearSVC Hyperparameter Tuning

In [ ]:
print_section('GridSearchCV - LinearSVC')

svm_base = LinearSVC(
    class_weight='balanced',
    max_iter=2000,
    random_state=42
)

svm_best, svm_best_params, svm_best_score, svm_cv_results = run_gridsearch(
    X_train, y_train,
    model=svm_base,
    param_grid={'C': [0.01, 0.1, 1.0, 10.0]},
    model_name='LinearSVC',
    cv=5
)

print(f'\nBest C     : {svm_best_params["C"]}')
print(f'Best CV F1 : {svm_best_score:.4f}')

### Random Forest and XGBoost Default Parameters

Random Forest and XGBoost have larger hyperparameter spaces
that would be computationally prohibitive to search on this
dataset. Standard configurations established in prior work
are used instead. Alghamdi et al. (2022) demonstrated that
these defaults perform competitively on text classification
tasks of similar scale.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)

print('Random Forest : n_estimators=100, class_weight=balanced')
print('XGBoost       : n_estimators=100, max_depth=6, lr=0.1')
print('Default parameters follow Alghamdi et al. (2022)')

## Section 5 -- Train Final Models

All four models are trained on the complete training set
using their optimal or default hyperparameters. Each model
is saved immediately after training so that results are
not lost if the session ends unexpectedly.

In [ ]:
print_section('Training All 4 Classical Models')

models = {
    'Logistic Regression': lr_best,
    'LinearSVC'          : svm_best,
    'Random Forest'      : rf_model,
    'XGBoost'            : xgb_model
}

model_filenames = {
    'Logistic Regression': 'model_lr.joblib',
    'LinearSVC'          : 'model_svm.joblib',
    'Random Forest'      : 'model_rf.joblib',
    'XGBoost'            : 'model_xgb.joblib'
}

for name, model in models.items():
    print(f'Training {name}...')
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    fpath = os.path.join(MODELS_DIR, model_filenames[name])
    save_model(model, fpath)
    print(f'  Completed in {elapsed:.1f}s\n')

## Section 6 -- Evaluate on Test Set

Final evaluation uses the held-out test set which has not
been touched during any previous stage. This ensures the
reported metrics represent an unbiased estimate of
real-world performance.

Macro F1 is the primary metric because the class distribution
shifted to 55.25% fake / 44.75% real after deduplication.
Macro F1 weights both classes equally regardless of frequency.

In [ ]:
print_section('Evaluation on Test Set')

results_list = []

for name, model in models.items():
    y_pred = model.predict(X_test)

    # LinearSVC uses decision_function instead of predict_proba
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)

    metrics = compute_metrics(y_test, y_pred, y_score, name)
    results_list.append(metrics)

    print(f'{name}')
    print(f"  Accuracy : {metrics['Accuracy']:.4f}")
    print(f"  F1 Macro : {metrics['F1_Macro']:.4f}")
    print(f"  ROC-AUC  : {metrics['ROC_AUC']:.4f}")
    print()

results_df = build_results_table(results_list)
save_results(results_df,
    os.path.join(RESULTS_DIR, 'classical_results.csv'))

## Section 7 -- Feature Importance Analysis

Logistic Regression assigns a signed coefficient to every
TF-IDF feature. Features with large negative coefficients
are strongly associated with Fake class articles. Features
with large positive coefficients are strongly associated
with Real class articles.

These top features provide interpretability and support the
poster session narrative about what linguistic patterns
distinguish fake from real news in the WELFake dataset.

In [ ]:
print_section('Feature Importance - Logistic Regression')

feature_names = vectorizer.get_feature_names_out()
lr_coefs      = lr_best.coef_[0]

# Top 20 features for each class by coefficient magnitude
top_fake_idx = np.argsort(lr_coefs)[:20]
top_real_idx = np.argsort(lr_coefs)[-20:][::-1]

top_fake_features = [(feature_names[i], lr_coefs[i])
                     for i in top_fake_idx]
top_real_features = [(feature_names[i], lr_coefs[i])
                     for i in top_real_idx]

print('Top 20 features associated with Fake class:')
for feat, coef in top_fake_features:
    print(f'  {feat:<35} {coef:.4f}')

print('\nTop 20 features associated with Real class:')
for feat, coef in top_real_features:
    print(f'  {feat:<35} {coef:.4f}')

## Section 8 -- Visualizations

Four plots are generated to support the results and
methodology sections of the written report and the
Streamlit demonstration application.

### Plot 11 -- Top TF-IDF Features by Class

In [ ]:
fake_terms = [f[0] for f in top_fake_features]
fake_coefs = [abs(f[1]) for f in top_fake_features]
real_terms = [f[0] for f in top_real_features]
real_coefs = [f[1] for f in top_real_features]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

ax1.barh(range(20), fake_coefs[::-1],
         color='#E76F51', alpha=0.85)
ax1.set_yticks(range(20))
ax1.set_yticklabels(fake_terms[::-1], fontsize=10)
ax1.set_xlabel('Absolute Coefficient Value')
ax1.set_title('Top 20 Features -- Fake Class', fontsize=13)

ax2.barh(range(20), real_coefs[::-1],
         color='#2A9D8F', alpha=0.85)
ax2.set_yticks(range(20))
ax2.set_yticklabels(real_terms[::-1], fontsize=10)
ax2.set_xlabel('Coefficient Value')
ax2.set_title('Top 20 Features -- Real Class', fontsize=13)

fig.suptitle(
    'TF-IDF Feature Importance by Class -- Logistic Regression',
    fontsize=15, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(FIGURES_DIR, '11_tfidf_top_features.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()

### Plot 12 -- Model Comparison Bar Chart

In [ ]:
metrics_cols = ['Accuracy', 'Precision', 'Recall',
                'F1_Macro', 'ROC_AUC']
model_names  = results_df['Model'].tolist()
colors       = ['#2A9D8F', '#E76F51', '#E9C46A', '#264653']
x     = np.arange(len(metrics_cols))
width = 0.2

fig, ax = plt.subplots(figsize=(14, 6))

for i, (mname, color) in enumerate(zip(model_names, colors)):
    row    = results_df[results_df['Model'] == mname].iloc[0]
    values = [row[m] for m in metrics_cols]
    bars   = ax.bar(x + i * width, values, width,
                    label=mname, color=color, alpha=0.85)
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f'{val:.3f}',
            ha='center', va='bottom', fontsize=8
        )

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics_cols, fontsize=11)
ax.set_ylabel('Score', fontsize=12)
ax.set_ylim(0.80, 1.02)
ax.set_title('Classical Model Performance Comparison',
             fontsize=14)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(
    os.path.join(FIGURES_DIR, '12_model_comparison.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()

### Plot 13 -- ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
colors_roc = ['#2A9D8F', '#E76F51', '#E9C46A', '#264653']

for (name, model), color in zip(models.items(), colors_roc):
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)

    fpr, tpr, _ = roc_curve(y_test, y_score)
    auc_val     = roc_auc_score(y_test, y_score)
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f'{name} (AUC = {auc_val:.4f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1,
        alpha=0.5, label='Random classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves -- Classical Models', fontsize=14)
ax.legend(fontsize=10, loc='lower right')
plt.tight_layout()
plt.savefig(
    os.path.join(FIGURES_DIR, '13_roc_curves.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()

### Plot 14 -- Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
class_labels = ['Fake (0)', 'Real (1)']

for idx, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test)
    cm     = confusion_matrix(y_test, y_pred)
    cm_pct = (cm.astype(float)
               / cm.sum(axis=1, keepdims=True) * 100)

    sns.heatmap(
        cm_pct, annot=True, fmt='.1f',
        cmap='YlOrRd',
        xticklabels=class_labels,
        yticklabels=class_labels,
        ax=axes[idx], cbar=False
    )

    for i in range(2):
        for j in range(2):
            axes[idx].text(
                j + 0.5, i + 0.72,
                f'n={cm[i, j]:,}',
                ha='center', va='center',
                fontsize=9, color='gray'
            )

    axes[idx].set_title(name, fontsize=12,
                        fontweight='bold')
    axes[idx].set_ylabel('True Label', fontsize=10)
    axes[idx].set_xlabel('Predicted Label', fontsize=10)

fig.suptitle(
    'Confusion Matrices -- Classical Models (% of true class)',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(FIGURES_DIR, '14_confusion_matrices.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()

## Section 9 -- Summary

In [ ]:
print_section('TF-IDF CLASSICAL BASELINE SUMMARY')

print('Optimal TF-IDF Configuration:')
print(f'  max_features : {OPTIMAL_MAX_FEATURES:,}')
print(f'  ngram_range  : {OPTIMAL_NGRAM}')
print(f'  min_df       : {OPTIMAL_MIN_DF}')
print(f'  max_df       : {OPTIMAL_MAX_DF}')
print()
print('Best Classifier Hyperparameters:')
print(f"  Logistic Regression C : {lr_best_params['C']}")
print(f"  LinearSVC C           : {svm_best_params['C']}")
print()
print('Final Test Set Results:')
print(results_df.to_string(index=False))
print()
best_model = results_df.iloc[0]['Model']
best_f1    = results_df.iloc[0]['F1_Macro']
print(f'Best model    : {best_model}')
print(f'Best F1 Macro : {best_f1:.4f}')

## Next Steps

The classical baseline results establish a performance ceiling
for TF-IDF based models. The next notebook builds a BiLSTM
neural baseline that captures sequential context and semantic
patterns beyond what TF-IDF bag-of-words can represent.

The best classical model F1 score serves as the primary
benchmark that the neural and hybrid models must exceed.